# ECSE 415 Course Project: Classification, Detection, and Localization

In [4]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os
import shutil
import random
import torch
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from pathlib import Path

## Part 1: The Classification Benchmark [50 Points]

Path declaration for part 1:

In [2]:
path = './part1/'

In [3]:
cats = Path(path+'/train/train/cats')
cat_imgs = list(cats.iterdir())
dogs = Path(path+'/train/train/dogs')
dog_imgs = list(dogs.iterdir())

#shuffle all cat and dog images
random.shuffle(cat_imgs)
random.shuffle(dog_imgs)

#split- first 80% -> training, last 20* of list -> internal testing
split_cats = int(len(cat_imgs) * 0.8)
split_dogs = int(len(dog_imgs) * 0.8)
train_cats = cat_imgs[:split_cats]
test_cats = cat_imgs[split_cats:]
train_dogs = dog_imgs[:split_dogs]
test_dogs = dog_imgs[split_dogs:]

#use shutil to copy images in each list 
for img in train_cats:
    shutil.copy2(img, path+'train/train_split/cats')
for img in test_cats:
    shutil.copy2(img, path+'train/test_split/cats')
for img in train_dogs:
    shutil.copy2(img, path+'train/train_split/dogs')
for img in test_dogs:
    shutil.copy2(img, path+'train/test_split/dogs')


### 1. Method Selection (Training)

**Option A**:

**Option B**:

**Option C**: Fine-Tuned ResNet

In [ ]:
# source for ResNet input information: https://pytorch.org/hub/pytorch_vision_resnet/
# source for justification of resnet 18: https://arxiv.org/pdf/1512.03385 

# Step 1: Load pre-trained ResNet-18 and modify the final layer for binary classification
# ResNet-18 is chosen over deeper variants (ResNet-34, ResNet-50) to reduce computational
# cost. Performance gains from deeper architectures are marginal for this binary task.
model = models.resnet18(pretrained=True)
num_classes = 2  # cats and dogs

# Replace the final fully connected layer to output 2 classes instead of ImageNet's 1000.
# in_features is preserved from the original layer.
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)


# Step 2: Preprocess training and testing data
# Apply data augmentation to training data to improve generalization and reduce overfitting.
# Test data is only transformed through normalization.
# Reference for normalization values: https://pytorch.org/hub/pytorch_vision_resnet/
data_transforms = {
    # Apply data augmentation and normalization to training data
    'train': transforms.Compose([
        # Randomly crop and resize to 224x224 to vary scale and position across epochs
        transforms.RandomResizedCrop(224), 
        # Randomly apply horizontal flip- this does not affect the fact that image still looks like a cat, and helps with augmentation
        transforms.RandomHorizontalFlip(),          
        transforms.ToTensor(),
        # Normalize using ImageNet mean and std required for pretrained ResNet weights
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])      
    ]),
    # Apply just resize+crop and normalization for test set
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}




## Part 2: Detection and Localization [50 Points]

In [4]:
path = './part2/'